# S2 polygon polyfill

Step-by-step animation of polygon → S2 cells, matching vgrid [`polygon2s2`](https://github.com/opengeoshub/vgrid/blob/main/vgrid/conversion/vector2dggs/vector2s2.py).

S2 uses `RegionCoverer` per part bbox (not BFS). Optional `compact=True` in `polygon2s2` is not animated here.

Input: [`multipolygon.geojson`](https://raw.githubusercontent.com/opengeoshub/vopendata/main/shape/multipolygon.geojson) — **12 features**, **13 polygon parts**. **S2 resolution** is shown on every frame.

## Install necessary packages

In [ ]:
%pip install vgrid geopandas matplotlib imageio pillow
# optional for MP4:
%pip install imageio-ffmpeg

In [ ]:
"""Step-by-step polygon2s2 animation (multipolygon.geojson)."""
from pathlib import Path

import geopandas as gpd
import imageio.v2 as imageio
import matplotlib.pyplot as plt
from matplotlib.collections import PatchCollection
from matplotlib.patches import Polygon as MplPolygon
from shapely.geometry import MultiPolygon, box

from vgrid.conversion.dggs2geo.s22geo import s22geo
from vgrid.conversion.vector2dggs.vector2s2 import polygon2s2
from vgrid.dggs import s2
from vgrid.utils.geometry import check_predicate

URL = "https://raw.githubusercontent.com/opengeoshub/vopendata/main/shape/multipolygon.geojson"
RESOLUTION = 14
PREDICATE = "intersects"
OUT_GIF = "polygon2s2.gif"
OUT_MP4 = "polygon2s2.mp4"
FRAME_EVERY_N = 8
DPI = 120
PART_COLORS = ["#1f4e79", "#c55a11", "#2e7d32", "#b71c1c", "#6a1b9a", "#4e342e", "#00695c", "#5d4037"]


def cell_patches(cell_polys, facecolor, edgecolor, alpha=0.55, lw=0.4):
    patches = []
    for poly in cell_polys:
        if poly is None or poly.is_empty:
            continue
        patches.append(MplPolygon(list(poly.exterior.coords), closed=True))
    return PatchCollection(
        patches, facecolor=facecolor, edgecolor=edgecolor, alpha=alpha, linewidths=lw
    )


def polygons_from_feature(feature):
    if feature.geom_type == "Polygon":
        return [feature]
    if feature.geom_type == "MultiPolygon":
        return list(feature.geoms)
    return []


def polygons_from_gdf(gdf):
    parts = []
    for geom in gdf.geometry:
        if geom is None or geom.is_empty:
            continue
        parts.extend(polygons_from_feature(geom))
    return parts


def feature_from_gdf(gdf):
    parts = polygons_from_gdf(gdf)
    if not parts:
        raise ValueError("No polygon geometries found in input GeoJSON")
    if len(parts) == 1:
        return parts[0]
    return MultiPolygon(parts)


def part_color(part_index):
    return PART_COLORS[part_index % len(PART_COLORS)]


def render_frame(
    parts,
    bbox,
    active_cells,
    title,
    path,
    resolution,
    current_poly=None,
    active_part=None,
):
    fig, ax = plt.subplots(figsize=(8, 8))
    union = MultiPolygon(parts) if len(parts) > 1 else parts[0]
    minx, miny, maxx, maxy = union.bounds
    pad = max(maxx - minx, maxy - miny) * 0.08 or 0.01
    ax.set_xlim(minx - pad, maxx + pad)
    ax.set_ylim(miny - pad, maxy + pad)

    for i, poly in enumerate(parts):
        color = part_color(i)
        lw = 3.0 if active_part == i else 1.8
        alpha = 1.0 if active_part is None or active_part == i else 0.45
        gpd.GeoSeries([poly]).plot(
            ax=ax, facecolor="none", edgecolor=color, lw=lw, alpha=alpha
        )
    if bbox is not None:
        gpd.GeoSeries([bbox]).plot(
            ax=ax, facecolor="none", edgecolor="#ff7f0e", lw=1.5, linestyle="--"
        )

    visited = list(active_cells) if active_cells else []
    if current_poly is not None:
        visited = [
            p
            for p in visited
            if p is not current_poly and not p.equals(current_poly)
        ]
    if visited:
        ax.add_collection(cell_patches(visited, "#2ca02c", "#1a5f1a", alpha=0.45))
    if current_poly is not None:
        ax.add_collection(
            cell_patches([current_poly], "#ffcc00", "#cc8800", alpha=0.9, lw=2.5)
        )
    ax.plot([], [], color="#ffcc00", lw=4, label="current cell")
    ax.plot([], [], color="#2ca02c", lw=4, label="bbox covering")
    ax.legend(loc="upper right", fontsize=8)
    ax.text(
        0.02,
        0.98,
        f"S2 resolution: {resolution}",
        transform=ax.transAxes,
        fontsize=9,
        va="top",
        ha="left",
        bbox=dict(boxstyle="round", facecolor="white", alpha=0.9),
        zorder=6,
    )
    ax.set_title(title)
    ax.set_aspect("equal")
    ax.grid(True, alpha=0.25)
    fig.subplots_adjust(left=0.08, right=0.92, top=0.92, bottom=0.08)
    fig.savefig(path, dpi=DPI, facecolor="white")
    plt.close(fig)


def polygon2s2_with_frames(parts, feature, resolution, predicate, frame_dir):
    frame_dir.mkdir(parents=True, exist_ok=True)
    frames = []
    idx = 0
    merged_tokens = []
    merged_polys = []
    seen_tokens = set()
    accumulated = []

    def snap(title, bbox=None, active=None, current=None, active_part=None):
        nonlocal idx
        p = frame_dir / f"frame_{idx:04d}.png"
        render_frame(
            parts,
            bbox,
            active if active is not None else accumulated,
            title,
            p,
            resolution,
            current_poly=current,
            active_part=active_part,
        )
        frames.append(p)
        idx += 1

    n_parts = len(parts)
    snap(
        f"1. Input res {resolution} ({n_parts} polygon part{'s' if n_parts != 1 else ''})"
    )
    snap("2. All parts (distinct colors)")

    coverer = s2.RegionCoverer()
    coverer.min_level = resolution
    coverer.max_level = resolution

    for part_i, polygon in enumerate(parts, start=1):
        min_lng, min_lat, max_lng, max_lat = polygon.bounds
        bbox = box(min_lng, min_lat, max_lng, max_lat)
        part_idx = part_i - 1
        region = s2.LatLngRect(
            s2.LatLng.from_degrees(min_lat, min_lng),
            s2.LatLng.from_degrees(max_lat, max_lng),
        )

        snap(
            f"3. Part {part_i}/{n_parts}: RegionCoverer level {resolution}",
            bbox=bbox,
            active_part=part_idx,
        )

        covering = list(coverer.get_covering(region))
        candidates = {}
        for step, cell_id in enumerate(covering, start=1):
            token = cell_id.to_token()
            cell_poly = s22geo(token)
            if cell_poly is None or cell_poly.is_empty:
                continue
            candidates[token] = cell_poly
            if step % FRAME_EVERY_N == 0 or step == len(covering):
                snap(
                    f"4. Part {part_i} covering {step}/{len(covering)}: {token}",
                    bbox=bbox,
                    active=list(candidates.values()),
                    current=cell_poly,
                    active_part=part_idx,
                )

        snap(
            f"5. Part {part_i} covering complete ({len(candidates)} candidates)",
            bbox=bbox,
            active=list(candidates.values()),
            active_part=part_idx,
        )

        part_final = []
        for token, cell_poly in candidates.items():
            if check_predicate(cell_poly, polygon, predicate):
                part_final.append((token, cell_poly))
                if token not in seen_tokens:
                    seen_tokens.add(token)
                    merged_tokens.append(token)
                    merged_polys.append(cell_poly)
                    accumulated.append(cell_poly)
        snap(
            f"6. Part {part_i} after predicate '{predicate}' ({len(part_final)} cells)",
            bbox=bbox,
            active=[p for _, p in part_final],
            active_part=part_idx,
        )

    snap(
        f"7. Merged result ({len(merged_tokens)} cells across {n_parts} parts)",
        active=merged_polys,
    )

    from_poly = []
    for part in parts:
        rows = polygon2s2(part, resolution, predicate=predicate)
        from_poly.extend(row["s2"] for row in rows)
    if set(from_poly) != set(merged_tokens):
        print(
            "Warning: cell set differs from polygon2s2:",
            len(merged_tokens),
            "vs",
            len(from_poly),
        )

    return frames, merged_tokens


def main():
    gdf = gpd.read_file(URL)
    parts = polygons_from_gdf(gdf)
    feature = feature_from_gdf(gdf)
    print(f"Loaded {len(gdf)} feature(s), {len(parts)} polygon part(s)")

    frame_dir = Path("_polygon2s2_frames")
    frames, cell_tokens = polygon2s2_with_frames(
        parts, feature, RESOLUTION, PREDICATE, frame_dir
    )
    imageio.mimsave(OUT_GIF, [imageio.imread(f) for f in frames], duration=0.9)
    print(f"Wrote {OUT_GIF} ({len(frames)} frames, {len(cell_tokens)} final cells)")
    try:
        writer = imageio.get_writer(OUT_MP4, fps=1.2)
        for f in frames:
            writer.append_data(imageio.imread(f))
        writer.close()
        print(f"Wrote {OUT_MP4}")
    except Exception as e:
        print(f"MP4 skipped ({e}). GIF is enough.")


if __name__ == "__main__":
    main()
